# Experiment 2  Shor's Factoring Algorithm
Factors N = 15 using quantum period finding. Classical post-processing with GCD gives factors 3 and 5.

In [ ]:
!pip install qiskit qiskit-aer matplotlib -q

## Imports

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
from math import gcd, pi
from fractions import Fraction
import numpy as np
import matplotlib.pyplot as plt

## Step 1  Classical Period Finding (what quantum replaces)
Shows f(x) = a^x mod N table so we understand what period r means.

In [ ]:
def show_period_table(base, modulus):
    print(f"Period finding: f(x) = {base}^x mod {modulus}")
    print(f"{'x':>4}  {'result':>8}")
    print("-" * 18)
    for x in range(1, modulus + 1):
        value = pow(base, x, modulus)
        print(f"{x:>4}  {value:>8}")
        if value == 1:
            print(f"\nPeriod r = {x}")
            return x

period = show_period_table(7, 15)

## Step 2 -Classical Post-Processing with GCD
Given period r and base a, extract factors of N.

In [ ]:
def extract_factors(base, period, modulus):
    if period % 2 != 0:
        print("Period is odd. Cannot proceed. Try different base.")
        return None, None

    half_power = pow(base, period // 2, modulus)
    factor_one = gcd(half_power - 1, modulus)
    factor_two = gcd(half_power + 1, modulus)

    print(f"base={base}, period={period}, base^(r/2) mod N = {half_power}")
    print(f"gcd({half_power}-1, {modulus}) = gcd({half_power-1}, {modulus}) = {factor_one}")
    print(f"gcd({half_power}+1, {modulus}) = gcd({half_power+1}, {modulus}) = {factor_two}")

    if factor_one not in [1, modulus] and factor_two not in [1, modulus]:
        return factor_one, factor_two
    return None, None

f1, f2 = extract_factors(7, 4, 15)
print(f"\nFactors of 15: {f1} x {f2} = {f1*f2}")

## Step 3 -Controlled Modular Exponentiation Gate
Quantum gate for a^(2^k) mod 15. Hardcoded using SWAP gates for N=15.

In [ ]:
def controlled_mod15(base, power):
    gate = QuantumCircuit(4)
    for _ in range(power):
        if base in [2, 13]:
            gate.swap(0, 1); gate.swap(1, 2); gate.swap(2, 3)
        if base in [7, 8]:
            gate.swap(2, 3); gate.swap(1, 2); gate.swap(0, 1)
        if base in [4, 11]:
            gate.swap(1, 3); gate.swap(0, 2)
        if base in [7, 11, 13]:
            for q in range(4):
                gate.x(q)
    controlled = gate.to_gate().control(1)
    controlled.name = f"{base}^{power} mod 15"
    return controlled

## Step 4 -Inverse QFT
Converts phase in counting register to a readable number.

In [ ]:
def inverse_qft(num_qubits):
    circuit = QuantumCircuit(num_qubits)

    for i in range(num_qubits // 2):
        circuit.swap(i, num_qubits - i - 1)

    for j in range(num_qubits):
        for m in range(j):
            circuit.cp(-pi / (2 ** (j - m)), m, j)
        circuit.h(j)

    circuit.name = "QFT_dagger"
    return circuit

## Step 5 -Full Shor's Circuit

In [ ]:
def build_shors_circuit(base=7, modulus=15, counting_qubits=8):
    total = counting_qubits + 4
    circuit = QuantumCircuit(total, counting_qubits)

    for q in range(counting_qubits):
        circuit.h(q)

    circuit.x(counting_qubits)

    for q in range(counting_qubits):
        circuit.append(
            controlled_mod15(base, 2 ** q),
            [q] + list(range(counting_qubits, counting_qubits + 4))
        )

    circuit.append(inverse_qft(counting_qubits), range(counting_qubits))
    circuit.measure(range(counting_qubits), range(counting_qubits))
    return circuit

## Step 6 -Run and Extract Factors

In [ ]:
base, modulus = 7, 15
shor_circuit  = build_shors_circuit(base, modulus, counting_qubits=8)

machine  = AerSimulator()
compiled = transpile(shor_circuit, machine)
result   = machine.run(compiled, shots=2048).result()
data     = result.get_counts()

plot_histogram(data, title=f"Shor's Algorithm — N={modulus}, a={base}", figsize=(9, 4))
plt.tight_layout()
plt.show()

print(f"\nFactoring N={modulus} with base a={base}")
print("=" * 45)

found = set()
for bitstring in data:
    decimal   = int(bitstring, 2)
    phase     = decimal / 256
    fraction  = Fraction(phase).limit_denominator(modulus)
    r         = fraction.denominator
    fa, fb    = extract_factors(base, r, modulus)
    if fa and (fa, fb) not in found:
        print(f"Output={bitstring}  phase={phase:.3f}  r={r}  factors={fa}x{fb}")
        found.add((fa, fb))